# Exploratory Data Analysis & Feature Engineering

This notebook featurizes the dataset found in Notebook 1 to prepare it for modeling.

## Steps
1. Load data from CSV files
2. Featurize the dataset
3. Transform into a data frame
4. Check for errors and save

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set up plotting
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set up paths
PROJECT_ROOT = Path().cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'

## Load Data

In [2]:
df = pd.read_csv(DATA_DIR / 'tm_oxygen_binary_compounds.csv')
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Dataset loaded successfully!
Shape: (288, 10)
Columns: ['material_id', 'formula', 'transition_metal', 'energy_above_hull', 'volume', 'density', 'band_gap', 'is_stable', 'is_metal', 'num_materials']


## Featurize and turn into a Data Frame

In [3]:
import subprocess
import sys

# Install matminer if needed
try:
    import matminer
    print("matminer already installed")
except ImportError:
    print("Installing matminer...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "matminer"])
    print("matminer installed successfully")

from matminer.featurizers.composition import ElementProperty
from pymatgen.core.composition import Composition

# Initialize the Magpie featurizer
magpie_featurizer = ElementProperty.from_preset("magpie")

# Featurize the dataset with Magpie
print(f"Featurizing {len(df)} compounds with Magpie features...\n")

features_list = []
errors = []

for idx, row in df.iterrows():
    if idx % 100 == 0:
        print(f"Processing {idx}/{len(df)}...")
    
    try:
        # Parse the formula to get composition
        formula = row['formula']
        composition = Composition(formula)
        
        # Generate Magpie features
        magpie_features = magpie_featurizer.featurize(composition)
        
        # Create a row with material info + all 132 features
        feature_row = {
            'material_id': row['material_id'],
            'formula': row['formula'],
            'transition_metal': row['transition_metal'],
        }
        
        # Add each Magpie feature
        for feature_name, feature_value in zip(magpie_featurizer.feature_labels(), magpie_features):
            feature_row[feature_name] = feature_value
        
        # Add target/property columns
        feature_row['energy_above_hull'] = row['energy_above_hull']
        feature_row['volume'] = row['volume']
        feature_row['density'] = row['density']
        feature_row['band_gap'] = row['band_gap']
        feature_row['is_stable'] = row['is_stable']
        feature_row['is_metal'] = row['is_metal']
        
        features_list.append(feature_row)
        
    except Exception as e:
        errors.append({'formula': row['formula'], 'error': str(e)})
        print(f"  Error featurizing {row['formula']}: {e}")

print(f"\nFeaturization complete!")
print(f"Successfully featurized: {len(features_list)} compounds")
print(f"Errors: {len(errors)} compounds")

if errors:
    print(f"\nFailed compounds:")
    for err in errors[:5]:  # Show first 5 errors
        print(f"  {err['formula']}: {err['error']}")

matminer already installed
Featurizing 288 compounds with Magpie features...

Processing 0/288...
Processing 100/288...
Processing 200/288...

Featurization complete!
Successfully featurized: 288 compounds
Errors: 0 compounds


/opt/anaconda3/envs/matds/lib/python3.10/site-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)


In [4]:
# Create DataFrame with all features
df_features = pd.DataFrame(features_list)

print(f"Featurized dataset shape: {df_features.shape}")
print(f"Number of columns: {len(df_features.columns)}")
print(f"\nColumn breakdown:")
print(f"  - Metadata columns: 3 (material_id, formula, transition_metal)")
print(f"  - Magpie features: 132")
print(f"  - Property columns: 6 (energy_above_hull, volume, density, band_gap, is_stable, is_metal)")

print(f"\nFirst 10 columns:")
print(df_features.columns[:10].tolist())

print(f"\nFirst 5 rows (showing first 10 columns):")
print(df_features.iloc[:5, :10])

print(f"\nDataset info:")
print(df_features.info())

Featurized dataset shape: (288, 141)
Number of columns: 141

Column breakdown:
  - Metadata columns: 3 (material_id, formula, transition_metal)
  - Magpie features: 132
  - Property columns: 6 (energy_above_hull, volume, density, band_gap, is_stable, is_metal)

First 10 columns:
['material_id', 'formula', 'transition_metal', 'MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number', 'MagpieData mean Number', 'MagpieData avg_dev Number', 'MagpieData mode Number', 'MagpieData minimum MendeleevNumber']

First 5 rows (showing first 10 columns):
  material_id  formula transition_metal  MagpieData minimum Number  \
0  mp-1179108     ScO2               Sc                        8.0   
1  mp-1186987     ScO3               Sc                        8.0   
2   mp-644481      ScO               Sc                        8.0   
3   mp-775837    Sc2O3               Sc                        8.0   
4   mp-759946  Sc16O15               Sc                        8.0   

   Mag

## Check for errors and save

In [5]:
# Check for NaN values in features
nan_count = df_features.isnull().sum().sum()
print(f"Total NaN values in featurized dataset: {nan_count}")

if nan_count > 0:
    print("\nColumns with NaN values:")
    nan_cols = df_features.isnull().sum()
    print(nan_cols[nan_cols > 0])
    
    # Fill NaN with 0 (if any)
    df_features = df_features.fillna(0)
    print("Filled NaN values with 0")

print(f"\nFinal featurized dataset shape: {df_features.shape}")

Total NaN values in featurized dataset: 0

Final featurized dataset shape: (288, 141)


In [6]:
# Save the featurized dataset
output_file = DATA_DIR / 'tm_oxygen_binary_compounds_magpie_features.csv'
df_features.to_csv(output_file, index=False)
print(f"Saved featurized dataset to {output_file}")

# Also save feature names for reference
feature_names = list(magpie_featurizer.feature_labels())
feature_file = DATA_DIR / 'magpie_feature_names.txt'
with open(feature_file, 'w') as f:
    for i, name in enumerate(feature_names, 1):
        f.write(f"{i}. {name}\n")
print(f"Saved feature names to {feature_file}")

print(f"\nSummary:")
print(f"  Input compounds: {len(df)}")
print(f"  Successfully featurized: {len(df_features)}")
print(f"  Features per compound: 132 Magpie + 6 properties = 138 total")

Saved featurized dataset to /Users/benbenmerk/Downloads/Merkin_FinalProject-main_test/data/tm_oxygen_binary_compounds_magpie_features.csv
Saved feature names to /Users/benbenmerk/Downloads/Merkin_FinalProject-main_test/data/magpie_feature_names.txt

Summary:
  Input compounds: 288
  Successfully featurized: 288
  Features per compound: 132 Magpie + 6 properties = 138 total
